# Embedding Space Visualization — AI-3 Session 1.2

We're going to watch ONE question travel through three retrieval strategies, in the geometry of embedding space.

**Focus query:** `"What are the performance review procedures?"`

**What we'll see, layer by layer:**
1. The corpus — 173 chunks, the *terrain*
2. Where the query lands — nowhere near the chunks it needs
3. What we wanted to retrieve — gold stars on the right chunks
4. HyDE — teleport the *query* into answer-space
5. Enrichment — plant question-shaped signposts *next to* every chunk

**Important:** 3D positions are PCA-projected. All similarity scores come from the full 1536-D embeddings — the 3D plot is illustrative, the numbers are authoritative.


In [ ]:
# Path setup + load the pre-computed embedding cache.
# If this cell errors on pickle, regenerate with:
#   uv run python notebooks/scripts/build_embedding_cache.py

import sys, os, pickle
from pathlib import Path

repo_root = str(Path().resolve().parent)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
os.chdir(repo_root)

import numpy as np
import plotly.graph_objects as go

CACHE_PATH = Path("notebooks/_cache/pca_viz_cache.pkl")
with CACHE_PATH.open("rb") as f:
    cache = pickle.load(f)

print(f"Loaded cache: {CACHE_PATH}")
print(f"  Corpus chunks: {len(cache["chunk_texts"])}")
print(f"  Queries: {len(cache["query_texts"])}")
print(f"  Enrichment questions: {len(cache["enrich_questions"])}")
print(f"  PCA explained variance per axis: {cache["pca"].explained_variance_ratio_}")


In [ ]:
# Helper utilities used by every plot cell below.

# Color / shape grammar for the whole notebook:
#   - Corpus chunks (unrelated): small pale gray circles
#   - Expected source chunks:    gold STARS
#   - User query (raw):          royal blue DIAMOND
#   - HyDE vector:               purple DIAMOND
#   - Enrichment questions (relevant): bright emerald CIRCLES
#   - Enrichment questions (manifold): faded emerald CIRCLES (low opacity)
COLORS = {
    "corpus":  "#D0D5DD",
    "gold":    "#F5B800",
    "query":   "#1D4ED8",
    "hyde":    "#7C3AED",
    "enrich":  "#10B981",
}


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    """Full-dimensional cosine similarity (authoritative score)."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def avg_cosine_to_chunks(query_vec: np.ndarray, chunk_vecs: np.ndarray) -> float:
    """Mean cosine similarity between a query and a set of target chunks."""
    return float(np.mean([cosine(query_vec, c) for c in chunk_vecs]))


def get_focus_query_index(id_: str) -> int:
    """Find which row in the cache corresponds to a given golden-set id."""
    return cache["query_ids"].index(id_)


def get_expected_chunk_mask(query_idx: int) -> np.ndarray:
    """Boolean mask over cache.chunk_* arrays: True where the chunk is an expected source."""
    expected = set(cache["query_expected_sources"][query_idx])
    return np.array([src in expected for src in cache["chunk_sources"]])


def base_layout(title: str) -> dict:
    """Shared 3D scene config — suppresses gridlines and tick clutter."""
    return dict(
        title=dict(text=title, x=0.5, xanchor="center"),
        scene=dict(
            xaxis=dict(title="PC1", showgrid=True, gridcolor="#EEE", backgroundcolor="white"),
            yaxis=dict(title="PC2", showgrid=True, gridcolor="#EEE", backgroundcolor="white"),
            zaxis=dict(title="PC3", showgrid=True, gridcolor="#EEE", backgroundcolor="white"),
            aspectmode="cube",
        ),
        margin=dict(l=0, r=0, t=40, b=0),
        height=600,
        showlegend=True,
        legend=dict(x=0.02, y=0.98, bgcolor="rgba(255,255,255,0.7)"),
    )


print("Helpers ready.")


## Step 1 — Pick the focus query

We're demo-ing `performance_review`:

> *"What are the performance review procedures?"*

**Expected source:** `employee_handbook.md` (gold stars).

**Why this query:** on the Session 1.1 scorecard, naive pulled `expense_policy.md` at rank 3 — a totally wrong-topic chunk (expenses, not performance reviews). HyDE and enrichment both recover. The bad-context story is visible in geometry.


In [ ]:
FOCUS_ID = "performance_review"
fq_idx = get_focus_query_index(FOCUS_ID)

# Masks and vectors we'll reuse across every plot cell
expected_mask = get_expected_chunk_mask(fq_idx)
expected_chunk_vecs = cache["chunk_emb"][expected_mask]

query_vec = cache["query_emb"][fq_idx]
hyde_vec = cache["hyde_emb"][fq_idx]

# Authoritative full-D cosine scores (NOT 3D distances)
cos_naive = avg_cosine_to_chunks(query_vec, expected_chunk_vecs)
cos_hyde  = avg_cosine_to_chunks(hyde_vec,  expected_chunk_vecs)

print(f"Focus query: {cache["query_texts"][fq_idx]}")
print(f"Expected sources: {cache["query_expected_sources"][fq_idx]}")
print(f"Expected chunks in corpus: {int(expected_mask.sum())}")
print()
print(f"Cosine (raw query ↔ expected chunks):  {cos_naive:.3f}")
print(f"Cosine (HyDE vector ↔ expected chunks): {cos_hyde:.3f}")


## Step 2 — The Terrain

Here is the full Northbrook corpus. Every dot is one chunk. These are the things retrieval has to choose from.

Nothing is highlighted yet. No queries, no answers — just the neighborhood.


In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=cache["chunk_proj"][:, 0],
    y=cache["chunk_proj"][:, 1],
    z=cache["chunk_proj"][:, 2],
    mode="markers",
    marker=dict(size=3, color=COLORS["corpus"], opacity=0.5),
    name="Corpus chunks (173)",
    hovertext=[f"{s}<br>{t[:120]}..." for s, t in zip(cache["chunk_sources"], cache["chunk_texts"])],
    hoverinfo="text",
))

fig.update_layout(**base_layout("The Terrain — 173 Corpus Chunks"))
fig.show()


## Step 3 — Where the Query Lands

Now the user's question enters the picture: a single blue diamond.

Watch which chunks are **nearest** to it. Are those chunks about performance reviews? That's where naive retrieval would pull from.


In [ ]:
fig = go.Figure()

# Corpus terrain
fig.add_trace(go.Scatter3d(
    x=cache["chunk_proj"][:, 0], y=cache["chunk_proj"][:, 1], z=cache["chunk_proj"][:, 2],
    mode="markers",
    marker=dict(size=3, color=COLORS["corpus"], opacity=0.5),
    name="Corpus chunks",
    hovertext=[f"{s}" for s in cache["chunk_sources"]],
    hoverinfo="text",
))

# Focus query — one big blue diamond
q = cache["query_proj"][fq_idx]
fig.add_trace(go.Scatter3d(
    x=[q[0]], y=[q[1]], z=[q[2]],
    mode="markers",
    marker=dict(size=10, color=COLORS["query"], symbol="diamond", line=dict(color="white", width=1)),
    name="User query",
    hovertext=[cache["query_texts"][fq_idx]],
    hoverinfo="text",
))

fig.update_layout(**base_layout("The Query Lands in Question-Space"))
fig.show()


## Step 4 — The Chunks We Actually Want

Now we mark the **correct** chunks (gold stars) — the ones from `employee_handbook.md` that contain the performance review policy.

Look at the geometry: the blue diamond is in one neighborhood, the gold stars are in another. **That gap is the embedding gap.** It's why naive retrieval pulls wrong-topic chunks — the nearest neighbors to the query are not the gold stars; they're other gray chunks from unrelated topics.


In [ ]:
fig = go.Figure()

# Non-expected corpus chunks (gray)
non_expected = ~expected_mask
fig.add_trace(go.Scatter3d(
    x=cache["chunk_proj"][non_expected, 0],
    y=cache["chunk_proj"][non_expected, 1],
    z=cache["chunk_proj"][non_expected, 2],
    mode="markers",
    marker=dict(size=3, color=COLORS["corpus"], opacity=0.4),
    name="Corpus chunks (unrelated)",
    hovertext=[f"{s}" for s in np.array(cache["chunk_sources"])[non_expected]],
    hoverinfo="text",
))

# Expected chunks — gold stars
fig.add_trace(go.Scatter3d(
    x=cache["chunk_proj"][expected_mask, 0],
    y=cache["chunk_proj"][expected_mask, 1],
    z=cache["chunk_proj"][expected_mask, 2],
    mode="markers",
    marker=dict(size=9, color=COLORS["gold"], symbol="diamond-open", line=dict(color=COLORS["gold"], width=3)),
    name=f"Expected chunks ({cache["query_expected_sources"][fq_idx][0]})",
    hovertext=[f"★ {s}<br>{t[:120]}..." for s, t in
               zip(np.array(cache["chunk_sources"])[expected_mask],
                   np.array(cache["chunk_texts"])[expected_mask])],
    hoverinfo="text",
))

# Focus query
q = cache["query_proj"][fq_idx]
fig.add_trace(go.Scatter3d(
    x=[q[0]], y=[q[1]], z=[q[2]],
    mode="markers",
    marker=dict(size=10, color=COLORS["query"], symbol="diamond", line=dict(color="white", width=1)),
    name="User query",
    hovertext=[cache["query_texts"][fq_idx]],
    hoverinfo="text",
))

fig.update_layout(**base_layout(
    f"The Gap — Query vs Expected Chunks (cosine = {cos_naive:.3f})"
))
fig.show()

print(f"Naive cosine to expected chunks: {cos_naive:.3f}")
print(f"(Anything below ~0.5 means retrieval will probably pull wrong-topic chunks first)")


## Step 5 — HyDE: Teleport the Query

HyDE asks Claude to *write a hypothetical answer* to the question, then embeds **that** instead of the raw question. The hypothetical answer is stylistically a policy document, so it lands near real policy chunks.

Watch the purple diamond (HyDE vector). The dashed line shows the query "teleporting" from question-space into answer-space.

**The score improves** — not because we found a better chunk yet, but because we changed where we're *looking* for chunks.


In [ ]:
fig = go.Figure()

# Corpus + gold stars (as before)
fig.add_trace(go.Scatter3d(
    x=cache["chunk_proj"][~expected_mask, 0],
    y=cache["chunk_proj"][~expected_mask, 1],
    z=cache["chunk_proj"][~expected_mask, 2],
    mode="markers",
    marker=dict(size=3, color=COLORS["corpus"], opacity=0.4),
    name="Corpus chunks", hoverinfo="skip",
))
fig.add_trace(go.Scatter3d(
    x=cache["chunk_proj"][expected_mask, 0],
    y=cache["chunk_proj"][expected_mask, 1],
    z=cache["chunk_proj"][expected_mask, 2],
    mode="markers",
    marker=dict(size=9, color=COLORS["gold"], symbol="diamond-open", line=dict(color=COLORS["gold"], width=3)),
    name="Expected chunks", hoverinfo="skip",
))

# Query + HyDE + the arrow between them
q = cache["query_proj"][fq_idx]
h = cache["hyde_proj"][fq_idx]

fig.add_trace(go.Scatter3d(
    x=[q[0]], y=[q[1]], z=[q[2]],
    mode="markers",
    marker=dict(size=10, color=COLORS["query"], symbol="diamond"),
    name="User query (raw)",
    hovertext=[cache["query_texts"][fq_idx]], hoverinfo="text",
))

fig.add_trace(go.Scatter3d(
    x=[h[0]], y=[h[1]], z=[h[2]],
    mode="markers",
    marker=dict(size=10, color=COLORS["hyde"], symbol="diamond"),
    name="HyDE vector",
    hovertext=[f"HyDE: {cache["hyde_texts"][fq_idx][:200]}..."],
    hoverinfo="text",
))

# Dashed arrow from query → HyDE
fig.add_trace(go.Scatter3d(
    x=[q[0], h[0]], y=[q[1], h[1]], z=[q[2], h[2]],
    mode="lines",
    line=dict(color=COLORS["hyde"], width=4, dash="dash"),
    name="HyDE teleport",
    hoverinfo="skip", showlegend=False,
))

fig.update_layout(**base_layout(
    f"HyDE Teleports the Query — cosine {cos_naive:.3f} → {cos_hyde:.3f}"
))
fig.show()

delta = cos_hyde - cos_naive
direction = "improved" if delta > 0 else "regressed"
print(f"Naive cosine: {cos_naive:.3f}")
print(f"HyDE cosine:  {cos_hyde:.3f}  ({direction} by {abs(delta):.3f})")


## Step 6 — Enrichment: Plant Signposts

Enrichment takes the opposite approach. Instead of moving the query into answer-space, we plant **question-shaped signposts** next to every chunk.

The bright green circles are the enrichment questions generated for the gold-star chunks. They sit right next to the blue diamond — because they're questions, and the user's query is a question. Questions match questions.

The faded green haze shows the **full enrichment manifold** — there are ~3 question-embeddings for every chunk in the corpus. Retrieval time, you land in question-space and the stored `documents` field points you back to the right chunk.


In [ ]:
fig = go.Figure()

# Corpus (non-expected, gray)
fig.add_trace(go.Scatter3d(
    x=cache["chunk_proj"][~expected_mask, 0],
    y=cache["chunk_proj"][~expected_mask, 1],
    z=cache["chunk_proj"][~expected_mask, 2],
    mode="markers",
    marker=dict(size=3, color=COLORS["corpus"], opacity=0.35),
    name="Corpus chunks", hoverinfo="skip",
))

# The enrichment manifold — all 519 question embeddings, faded
fig.add_trace(go.Scatter3d(
    x=cache["enrich_proj"][:, 0],
    y=cache["enrich_proj"][:, 1],
    z=cache["enrich_proj"][:, 2],
    mode="markers",
    marker=dict(size=2, color=COLORS["enrich"], opacity=0.12),
    name=f"Enrichment manifold ({len(cache["enrich_questions"])})",
    hoverinfo="skip",
))

# The enrichment questions generated for OUR expected chunks — bright
expected_chunk_ids = set(np.where(expected_mask)[0].tolist())
relevant_enrich_mask = np.array(
    [ci in expected_chunk_ids for ci in cache["enrich_chunk_indices"]]
)
fig.add_trace(go.Scatter3d(
    x=cache["enrich_proj"][relevant_enrich_mask, 0],
    y=cache["enrich_proj"][relevant_enrich_mask, 1],
    z=cache["enrich_proj"][relevant_enrich_mask, 2],
    mode="markers",
    marker=dict(size=7, color=COLORS["enrich"], opacity=1.0, line=dict(color="white", width=1)),
    name="Enrichment Qs for expected chunks",
    hovertext=[q for q, rel in zip(cache["enrich_questions"], relevant_enrich_mask) if rel],
    hoverinfo="text",
))

# Gold stars
fig.add_trace(go.Scatter3d(
    x=cache["chunk_proj"][expected_mask, 0],
    y=cache["chunk_proj"][expected_mask, 1],
    z=cache["chunk_proj"][expected_mask, 2],
    mode="markers",
    marker=dict(size=9, color=COLORS["gold"], symbol="diamond-open", line=dict(color=COLORS["gold"], width=3)),
    name="Expected chunks", hoverinfo="skip",
))

# Query + HyDE (keep them visible so students see all three strategies in one view)
q = cache["query_proj"][fq_idx]
h = cache["hyde_proj"][fq_idx]
fig.add_trace(go.Scatter3d(
    x=[q[0]], y=[q[1]], z=[q[2]], mode="markers",
    marker=dict(size=10, color=COLORS["query"], symbol="diamond"),
    name="User query", hoverinfo="skip",
))
fig.add_trace(go.Scatter3d(
    x=[h[0]], y=[h[1]], z=[h[2]], mode="markers",
    marker=dict(size=10, color=COLORS["hyde"], symbol="diamond"),
    name="HyDE vector", hoverinfo="skip",
))

fig.update_layout(**base_layout(
    "Enrichment — Question-Space Is Now the Index"
))
fig.show()

# Quick numerical readout
enrich_vecs_for_expected = cache["enrich_emb"][relevant_enrich_mask]
cos_enrich = avg_cosine_to_chunks(query_vec, enrich_vecs_for_expected)
print(f"Cosine (raw query ↔ enrichment Qs for expected chunks): {cos_enrich:.3f}")
print("(Note: enrichment retrieval is query↔enrichment-Q, then the matched Q points back to its chunk.)")


## Recap — Two Techniques, Two Geometries

| Technique | What moves | Where it lands | How retrieval works |
|---|---|---|---|
| **Naive** | nothing | raw query stays in question-space | nearest-neighbor search from query into corpus → often wrong-topic neighbors |
| **HyDE** | the *query* | moves into answer-space (via hypothetical answer) | search from HyDE vector into corpus → lands near real answers |
| **Enrichment** | the *index* | plants question-shaped signposts next to every chunk | search from raw query into **question-space**, then map back to chunk |

Both close the gap from opposite sides. Naive is stuck in the wrong neighborhood by default.

**Why we needed to see this:** the DR-012 bug from Monday was an embedding that **belonged to neither** neighborhood — combined chunk+question text that landed in no-man's-land. That bug is invisible until you either (a) look at the scorecard and see low scores or (b) look at the geometry and see the vectors floating between clusters. **That's why evaluation exists.**

---

## Bonus — Try It Yourself

Swap `FOCUS_ID` in Step 1 for any of these golden-set IDs and re-run cells 3 onward:

- `ceo_priorities` — naive lands in the retrospective blurb, HyDE actually **regresses** here (hallucinated a merger), enrichment wins
- `q4_board_meeting` — HyDE was originally the winner but hallucinated; enrichment caught the real doc
- `office_move_timeline` — compound question; enrichment anchors all three expected chunks
- `expense_reimbursement` — single-topic lookup; HyDE dominates

Notice how the geometry changes per query. The technique that wins depends on the *shape* of the question, which is why we need evaluation rather than intuition.


In [ ]:
# Re-run with a different focus query — try "ceo_priorities" to see HyDE regress.
# Change the value below, then re-run cells starting from Step 1 (the `fq_idx = ...` cell).

ALTERNATE_FOCUS = "ceo_priorities"

# Quick check: does the alternate query exist in the cache?
if ALTERNATE_FOCUS in cache["query_ids"]:
    alt_idx = cache["query_ids"].index(ALTERNATE_FOCUS)
    alt_mask = get_expected_chunk_mask(alt_idx)

    alt_cos_naive = avg_cosine_to_chunks(cache["query_emb"][alt_idx], cache["chunk_emb"][alt_mask])
    alt_cos_hyde  = avg_cosine_to_chunks(cache["hyde_emb"][alt_idx],  cache["chunk_emb"][alt_mask])

    print(f"ALTERNATE FOCUS: {cache["query_texts"][alt_idx]}")
    print(f"  Expected sources: {cache["query_expected_sources"][alt_idx]}")
    print(f"  Naive cosine:  {alt_cos_naive:.3f}")
    print(f"  HyDE cosine:   {alt_cos_hyde:.3f}  <-- notice if this one regresses!")
    print()
    print(f"To explore: set FOCUS_ID = '{ALTERNATE_FOCUS}' in Step 1 and re-run.")
else:
    print(f"'{ALTERNATE_FOCUS}' not in golden set. Available: {cache["query_ids"]}")
